# Column Name and Data Format Standardization

**Learning Unit: Data Cleaning Fundamentals**  
**Date: March 2026**

---

## Learning Objectives

By completing this notebook, you will be able to:

1. ✅ Convert column names to a consistent format
2. ✅ Remove spaces and special characters from column names
3. ✅ Apply predictable naming conventions (snake_case)
4. ✅ Standardize simple data formats across columns
5. ✅ Improve dataset usability and readability

---

## Why Standardization Matters

**Common beginner issues:**
- Column names with spaces or mixed casing
- Inconsistent naming across datasets
- Difficulty referencing columns in code
- Errors when merging or transforming data

**Messy column names lead to messy code.**

Standardization ensures:
- Column access is simple and predictable
- Code is cleaner and less error-prone
- Datasets are easier to merge and reuse
- Analysis workflows scale better

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import re

## Part 1: Create Sample Data with Messy Column Names

Let's create a dataset with intentionally messy column names to demonstrate common issues.

In [ ]:
# Create sample data with messy column names
data = {
    'Employee ID': [1001, 1002, 1003, 1004, 1005, 1006],
    'Department Name': ['  Engineering', 'marketing  ', 'Sales', 'hr', 'FINANCE', 'Engineering'],
    'Survey Date (Q1)': ['2026-01-15', '2026-01-16', '2026-01-17', '2026-01-18', '2026-01-19', '2026-01-20'],
    'Satisfaction Score (1-10)': [7, 9, 4, 8, 6, 9],
    'Work/Life Balance': [8, 9, 3, 7, 7, 8],
    'Manager Support %': [6, 8, 4, 8, 5, 9],
    'Career-Growth': [5, 7, 3, 7, 4, 8],
    'Team Collaboration!!!': [9, 8, 5, 9, 6, 10]
}

df_messy = pd.DataFrame(data)

print("Dataset created with messy column names!")
print(f"Shape: {df_messy.shape}")
print("\nOriginal Column Names:")
for i, col in enumerate(df_messy.columns, 1):
    print(f"{i}. '{col}'")

In [ ]:
# Display the messy DataFrame
df_messy

## Part 2: Identify Column Name Issues

Let's identify what's wrong with these column names.

In [ ]:
# Identify issues in column names
print("Column Name Issues Found:\n")

for col in df_messy.columns:
    issues = []
    
    if ' ' in col:
        issues.append("contains spaces")
    if col != col.lower():
        issues.append("mixed/upper case")
    if any(char in col for char in ['(', ')', '%', '/', '!', '-']):
        issues.append("special characters")
    
    if issues:
        print(f"❌ '{col}'")
        for issue in issues:
            print(f"   • {issue}")
        print()

### Why These Issues Matter

❌ **Bad:**
```python
df['Employee ID']  # Requires quotes and spaces
df['Work/Life Balance']  # Special characters are confusing
```

✅ **Good:**
```python
df['employee_id']  # Clean, simple, predictable
df['work_life_balance']  # No special characters, easy to type
```

## Part 3: Standardize Column Names

Let's create a function to standardize column names using **snake_case** convention.

In [ ]:
def standardize_column_names(df):
    """
    Standardize column names to snake_case format.
    
    Rules:
    1. Convert to lowercase
    2. Replace spaces with underscores
    3. Remove special characters
    4. Ensure consistency
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame with potentially messy column names
        
    Returns:
    --------
    pd.DataFrame
        DataFrame with standardized column names
    """
    # Create a copy to avoid modifying original
    df_clean = df.copy()
    
    # Apply standardization rules
    new_columns = []
    for col in df_clean.columns:
        # Convert to lowercase
        col_clean = col.lower()
        
        # Replace spaces with underscores
        col_clean = col_clean.replace(' ', '_')
        
        # Remove special characters (keep only alphanumeric and underscore)
        col_clean = re.sub(r'[^a-z0-9_]', '', col_clean)
        
        # Remove multiple consecutive underscores
        col_clean = re.sub(r'_+', '_', col_clean)
        
        # Remove leading/trailing underscores
        col_clean = col_clean.strip('_')
        
        new_columns.append(col_clean)
    
    # Assign cleaned column names
    df_clean.columns = new_columns
    
    return df_clean

print("✅ Standardization function defined!")

In [ ]:
# Apply standardization
df_clean = standardize_column_names(df_messy)

print("Column Name Transformation:\n")
for orig, new in zip(df_messy.columns, df_clean.columns):
    print(f"'{orig}' → '{new}'")

In [ ]:
# Display cleaned DataFrame
df_clean

In [ ]:
# Compare column names side by side
comparison_df = pd.DataFrame({
    'Original': df_messy.columns.tolist(),
    'Standardized': df_clean.columns.tolist()
})

comparison_df

## Part 4: Standardize Text Data Formats

Now let's standardize the actual data values, especially text columns.

In [ ]:
# Inspect text data BEFORE standardization
print("Department column BEFORE standardization:")
print(df_clean['department_name'].values)
print("\nNotice the issues:")
print("• Leading/trailing whitespace")
print("• Inconsistent casing (lowercase, UPPERCASE, Title Case)")

In [ ]:
def standardize_text_columns(df, text_columns):
    """
    Standardize text data in specified columns.
    
    Rules:
    - Strip whitespace
    - Consistent casing
    - Remove extra spaces
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame to clean
    text_columns : list
        List of column names to standardize
        
    Returns:
    --------
    pd.DataFrame
        DataFrame with standardized text columns
    """
    df_clean = df.copy()
    
    for col in text_columns:
        if col in df_clean.columns:
            # Strip leading/trailing whitespace
            df_clean[col] = df_clean[col].astype(str).str.strip()
            
            # Remove extra internal spaces
            df_clean[col] = df_clean[col].str.replace(r'\s+', ' ', regex=True)
            
            # For categorical columns, ensure consistent casing (title case)
            if col == 'department_name':
                df_clean[col] = df_clean[col].str.title()
    
    return df_clean

# Apply text standardization
df_final = standardize_text_columns(df_clean, ['department_name'])

print("Text standardization function applied!")

In [ ]:
# Inspect text data AFTER standardization
print("Department column AFTER standardization:")
print(df_final['department_name'].values)
print("\nImprovements:")
print("✅ Whitespace removed")
print("✅ Consistent Title Case format")
print("✅ Ready for grouping and analysis")

## Part 5: Before and After Comparison

In [ ]:
print("📊 BEFORE (Messy):")
display(df_messy.head(3))

print("\n" + "="*80 + "\n")

print("📊 AFTER (Standardized):")
display(df_final.head(3))

## Part 6: Practical Benefits

### Benefits of Standardized Column Names:

1. **Easier code:** `df['employee_id']` vs `df['Employee ID']`
2. **Predictable patterns** across datasets
3. **No errors** from spaces or special characters
4. **Cleaner merges** and joins
5. **Better autocomplete** in IDEs
6. **More professional** and maintainable code

### Benefits of Standardized Data Formats:

1. **Consistent grouping** and aggregation
2. **Reliable sorting** and filtering
3. **Fewer bugs** from data inconsistencies
4. **Easier data validation**
5. **Better downstream processing**

## Part 7: Practice with Real Data

In [ ]:
# Try loading the real employee survey data
try:
    df_survey = pd.read_csv('../data/raw/employee_survey_2026_Q1.csv')
    
    print("✓ Real dataset loaded successfully!")
    print(f"Shape: {df_survey.shape}")
    print("\nCurrent column names:")
    for i, col in enumerate(df_survey.columns, 1):
        print(f"{i}. {col}")
    
    # Check if it needs cleaning
    needs_cleaning = any(col != col.lower() or ' ' in col for col in df_survey.columns)
    
    if needs_cleaning:
        print("\n⚠️ This dataset has column naming issues!")
        print("Try applying the standardize_column_names() function to it.")
    else:
        print("\n✓ This dataset already follows good naming conventions!")
        print("This is what we aim for in our own datasets!")
        
except FileNotFoundError:
    print("⚠️ Real survey data file not found.")
    print("Using demonstration data instead.")

## Part 8: Your Turn - Practice Exercise

Create your own messy dataset and apply standardization!

In [ ]:
# TODO: Create a DataFrame with messy column names
# Example structure:
practice_data = {
    'First Name': ['Alice', 'Bob', 'Charlie'],
    'Last Name ': ['Smith', 'Jones', 'Brown'],
    'Age (years)': [25, 30, 35],
    'Salary$$$': [50000, 60000, 75000]
}

df_practice = pd.DataFrame(practice_data)

# Display original
print("Original:")
display(df_practice)

# Apply standardization
df_practice_clean = standardize_column_names(df_practice)

# Display cleaned
print("\nCleaned:")
display(df_practice_clean)

## Summary: Best Practices

### 🎯 Column Naming Convention (snake_case):
- ✅ **Good:** `employee_id`, `satisfaction_score`, `work_life_balance`
- ❌ **Avoid:** `Employee ID`, `Satisfaction Score`, `Work/Life Balance`

### 🎯 Text Data Standardization:
- ✅ **Good:** Consistent casing, no extra whitespace
- ❌ **Avoid:** Mixed casing, leading/trailing spaces

### 🎯 When to Standardize:
- Immediately after loading data
- Before any analysis or merging
- At the start of your data pipeline

### 🎯 Why It Matters:
- **Clean data = Clean code**
- Prevents errors and bugs
- Makes datasets reusable
- Enables scalable analysis

---

## 💡 Remember:

**Standardize early, standardize always!**

Your future self (and teammates) will thank you.

---

## Next Steps

1. ✅ Complete the video walkthrough (≈2 minutes)
2. ✅ Practice with your own datasets
3. ✅ Create a reusable standardization function
4. ✅ Apply standardization at the start of every project
5. ✅ Share these practices with your team